In [1]:
!pip install biopython

   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 31.7 MB/s eta 0:00:00


In [9]:
from Bio import Entrez
import pandas as pd
import json

Entrez.email = "your_email@example.com"

pmid_list = ["22260327"]

records = []

handle = Entrez.efetch(
    db="pubmed",
    id=",".join(pmid_list),
    rettype="abstract",
    retmode="xml"
)

data = Entrez.read(handle)

for article in data["PubmedArticle"]:

    citation = article["MedlineCitation"]
    art = citation["Article"]

    pmid = str(citation["PMID"])
    title = art.get("ArticleTitle", "")

    # Authors
    authors = []
    for a in art.get("AuthorList", []):
        if "LastName" in a:
            authors.append(f"{a.get('ForeName','')} {a.get('LastName','')}")

    # Abstract sections
    abstract_sections = []

    if "Abstract" in art:
        for i, sec in enumerate(art["Abstract"]["AbstractText"], start=1):

            label = sec.attributes.get("Label", "")

            abstract_sections.append({
                "section_id": i,
                "label": label,
                "text": str(sec)
            })

    records.append({
        "pmid": pmid,
        "title": title,
        "authors": "; ".join(authors),
        "abstract_sections": json.dumps(abstract_sections, ensure_ascii=False)
    })

df = pd.DataFrame(records)

df.to_csv("pubmed_articles.csv", index=False)